In [6]:
import ome_zarr
from ome_zarr.io import parse_url
from ome_zarr.writer import write_multiscale
from ome_zarr.writer import write_multiscales_metadata
import zarr

import dask.array as da
import numpy as np
import glob
from skimage import io
import tqdm
from tifffile import TiffFile
import yaml
import os
from pathlib import Path

In [7]:
# like numpy.mean, but maintains dtype
def mean_dtype(arr, **kwargs):
    return np.mean(arr, **kwargs).astype(arr.dtype)

In [8]:
root_folder = r'E:\Data\BiAPoL\Microwells\20250512\20250512_well1_BF_3Zplanes_full_1'
target = r'E:\Data\BiAPoL\Microwells\20250512\20250512_well1_BF_3Zplanes_full_1_zarrs'

In [9]:
metadata_files = glob.glob(f"{root_folder}/**/metadata.txt", recursive=True)
metadata_files[0]

with open(metadata_files[0], 'r') as f:
    metadata = yaml.safe_load(f)['Summary']

In [10]:
z_scale = metadata['z-step_um']
y_scale = metadata['PixelSize_um']
x_scale = metadata['PixelSize_um']
positionlist = metadata['InitialPositionList']
positionlist = {position['Label']: position for position in positionlist}

In [12]:
path = os.path.join(r'E:\Data\BiAPoL\Microwells\20250512\20250512_well1_BF_3Zplanes_full_1' + '.ome.zarr')
path

'E:\\Data\\BiAPoL\\Microwells\\20250512\\20250512_well1_BF_3Zplanes_full_1.ome.zarr'

In [13]:
folders = os.listdir(root_folder)
for i, folder in tqdm.tqdm(enumerate(folders), total=len(folders)):
    if not os.path.isdir(os.path.join(root_folder, folder)):
        continue

    # read image stack
    files = sorted(glob.glob(os.path.join(root_folder, folder, '*.tif')))
    array = da.stack([
        da.from_array(io.imread(f)) for f in files],
    )

    scales = [array]
    for _ in range(3):
        scales.append(
            da.coarsen(mean_dtype, scales[-1], {scales[-1].ndim - 2: 2, scales[-1].ndim - 1: 2}, trim_excess=True)
        )

    translate_x = float(
        positionlist[folder]["DeviceCoordinatesUm"]["XYStage:XY:31"][0]
    )
    translate_y = -float(
        positionlist[folder]["DeviceCoordinatesUm"]["XYStage:XY:31"][1]
    )
    translate_z = float(
        positionlist[folder]["DeviceCoordinatesUm"]["ZStage:Z:32"][0]
    )

    coordtfs = [
        [
            {
                'type': 'scale',
                'scale': [z_scale * (2 ** i), y_scale * (2 ** i), x_scale * (2 ** i)],
                },
            {
                'type': 'translation',
                'translation': [translate_z, translate_y, translate_x],
                }
        ]
        for i in range(4)
    ]

    axes = [
        {
            'name': 'z',
            'type': 'space',
            'unit': 'micrometer'
            },
        {
            'name': 'y',
            'type': 'space',
            'unit': 'micrometer'
            },
        {
            'name': 'x',
            'type': 'space',
            'unit': 'micrometer'
            }
    ]

    ome = {
        "channels": [
            {
                "active": True,
                "color": "ffffff",
                "label": folder,
                "window": {
                    "start": 7500,
                    "end": 35000,
                    "min": 0,
                    "max": 65535
                }
            }
        ]
    }
    break
    path = os.path.join(target, Path(folder).stem + '.ome.zarr')
    store = parse_url(path, mode="w").store
    root = zarr.group(store=store)

    ome_zarr.writer.write_multiscale(
        pyramid=scales,
        group=root,
        axes=axes,
        coordinate_transformations=coordtfs,
        name=folder,
    )
    root.attrs['omero'] = ome

  0%|          | 0/324 [00:00<?, ?it/s]


In [14]:
scales

[dask.array<stack, shape=(3, 1536, 1536), dtype=uint16, chunksize=(1, 1536, 1536), chunktype=numpy.ndarray>,
 dask.array<coarsen, shape=(3, 768, 768), dtype=uint16, chunksize=(1, 768, 768), chunktype=numpy.ndarray>,
 dask.array<coarsen, shape=(3, 384, 384), dtype=uint16, chunksize=(1, 384, 384), chunktype=numpy.ndarray>,
 dask.array<coarsen, shape=(3, 192, 192), dtype=uint16, chunksize=(1, 192, 192), chunktype=numpy.ndarray>]